# Lecture Notes Study Tutor — Q&A, Summaries & Quiz Mode

## 1. Project Overview

**Task:** Ingest a set of lecture notes (markdown/text format), build a retrieval index over them, and provide three interactive study modes:

| Mode | What It Does |
|------|--------------|
| **Q&A** | Answer student questions with cited lecture references |
| **Summary** | Generate concise summaries of any lecture or topic |
| **Quiz** | Generate multiple-choice and short-answer questions, then grade answers |

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM for all generation
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — local embeddings
- `ChromaDB` — vector store with lecture metadata

> **No API keys required.** Everything runs locally with Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Educational RAG design** — tailor retrieval and generation for learning contexts |
| 2 | **Lecture-aware chunking** — split notes by topic/section, not arbitrary boundaries |
| 3 | **Multi-mode generation** — same index serves Q&A, summarization, and quiz tasks |
| 4 | **Quiz generation** — create questions at varying difficulty from retrieved content |
| 5 | **Answer grading** — use LLM to evaluate student answers against source material |
| 6 | **Retrieval evaluation** — Recall@k and MRR on ground-truth question-section pairs |

## 3. Problem Statement

Students face several challenges when studying from lecture notes:

- **Finding relevant material** — scrolling through 200 slides to find the one concept you need
- **Active recall** — passively re-reading notes is ineffective; quizzing yourself works better
- **Summarization** — distilling key points from a verbose lecture into study-ready form
- **Self-assessment** — no way to test understanding without waiting for an exam

A study tutor that understands the lecture content can provide on-demand Q&A, generate summaries at the right granularity, and quiz students to reinforce learning.

## 4. Educational RAG Design Principles

Building RAG for education is different from building RAG for general search. Key design choices:

| Principle | General RAG | Educational RAG |
|-----------|------------|----------------|
| **Chunking** | Character/token-based | Topic/section-based — aligned with how lectures are organized |
| **Metadata** | Source doc only | Lecture number, topic, difficulty, concept type |
| **Retrieval** | Closest match wins | May need *related* material too (prerequisites, examples) |
| **Generation tone** | Neutral/professional | Pedagogical — explain, not just state |
| **Citing sources** | Optional | Essential — students need to know *where* to review |
| **Follow-up** | None | Quiz → check → explain wrong answers |

The key insight: an educational system should help students **understand**, not just find information.

## 5. Pipeline Architecture

```
Lecture Notes (organized by lecture number + topic)
       │
  ┌────┴──────────────────────────────────────────────┐
  │  Stage 1: PARSE & CHUNK                            │
  │  • Split by section headings                       │
  │  • Extract metadata: lecture #, topic, key terms    │
  │  • Track concept relationships                     │
  └────────────────────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────────┐
  │  Stage 2: EMBED & INDEX                            │
  │  • Embed with sentence-transformers                │
  │  • Store in ChromaDB with lecture metadata          │
  └────────────────────────────────────────────────────┘
       │
  ┌────┴────────────────────────────────┐
  │  Stage 3: STUDY MODES               │
  │  ┌─────────┬──────────┬───────────┐ │
  │  │  Q & A   │ Summary  │   Quiz    │ │
  │  │ answer   │ condense │ generate  │ │
  │  │ w/ cites │ by topic │ MCQ + SA  │ │
  │  └─────────┴──────────┴───────────┘ │
  └────────────────────────────────────┘
       │
  ┌────┴──────────────────────────────────────────────┐
  │  Stage 4: EVALUATE                                 │
  │  • Retrieval: Recall@k, MRR                        │
  │  • Answer quality: LLM-as-judge                    │
  │  • Quiz quality: difficulty calibration check       │
  └────────────────────────────────────────────────────┘
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers

print("Dependencies: langchain, langchain-ollama, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

## 8. Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from collections import Counter
from typing import Optional

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 5
TEMP_ANSWER = 0.1
TEMP_QUIZ = 0.4
TEMP_JUDGE = 0.0

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Retrieval top-k: {TOP_K}")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Synthetic Lecture Notes

## 10. Build the Lecture Corpus

We simulate a 6-lecture introductory Machine Learning course. Each lecture has multiple sections with key concepts, examples, and formulas.

In [ ]:
LECTURES = [
    {
        "lecture_num": 1,
        "title": "Introduction to Machine Learning",
        "topic": "foundations",
        "sections": [
            {"heading": "What is Machine Learning?",
             "content": "Machine learning is a subset of artificial intelligence that focuses on building systems that learn from data rather than being explicitly programmed. Instead of writing rules by hand, we provide examples and the algorithm discovers patterns. Arthur Samuel (1959) defined it as the field of study that gives computers the ability to learn without being explicitly programmed. Modern ML powers recommendation systems, image recognition, natural language processing, and autonomous vehicles.",
             "key_terms": ["machine learning", "artificial intelligence", "pattern recognition"],
             "difficulty": "beginner"},
            {"heading": "Types of Machine Learning",
             "content": "There are three main paradigms of ML. (1) Supervised learning: the algorithm learns from labeled examples — we provide both the input and the correct output. Examples: email spam detection, house price prediction. (2) Unsupervised learning: the algorithm finds structure in unlabeled data. Examples: customer segmentation, anomaly detection. (3) Reinforcement learning: an agent learns by interacting with an environment and receiving rewards or penalties. Examples: game playing, robotics. Semi-supervised learning blends the first two, using a small set of labeled data with a large pool of unlabeled data.",
             "key_terms": ["supervised learning", "unsupervised learning", "reinforcement learning"],
             "difficulty": "beginner"},
            {"heading": "The ML Pipeline",
             "content": "A typical ML project follows these steps: (1) Problem definition — what are we trying to predict or discover? (2) Data collection — gather relevant data from databases, APIs, or sensors. (3) Data preprocessing — clean missing values, handle outliers, encode categorical variables. (4) Feature engineering — create informative features from raw data. (5) Model selection — choose an algorithm appropriate for the task. (6) Training — fit the model to training data. (7) Evaluation — measure performance on held-out test data. (8) Deployment — serve predictions in production. Each step can iterate as you learn more about the problem.",
             "key_terms": ["pipeline", "feature engineering", "model selection", "deployment"],
             "difficulty": "beginner"},
        ],
    },
    {
        "lecture_num": 2,
        "title": "Linear Regression",
        "topic": "regression",
        "sections": [
            {"heading": "Simple Linear Regression",
             "content": "Simple linear regression models the relationship between a single independent variable x and a dependent variable y as a straight line: y = wx + b, where w is the slope (weight) and b is the y-intercept (bias). The goal is to find w and b that minimize the difference between predicted and actual values. For example, predicting house price (y) from square footage (x). The model assumes a linear relationship, constant variance of errors (homoscedasticity), and independence of observations.",
             "key_terms": ["linear regression", "slope", "intercept", "homoscedasticity"],
             "difficulty": "beginner"},
            {"heading": "Cost Function and Gradient Descent",
             "content": "The Mean Squared Error (MSE) cost function measures how wrong our predictions are: MSE = (1/n) * sum((y_pred - y_actual)^2). We want to minimize this. Gradient descent is an optimization algorithm that iteratively adjusts w and b in the direction that reduces MSE. The update rules are: w = w - learning_rate * dMSE/dw and b = b - learning_rate * dMSE/db. The learning rate controls step size — too large and we overshoot, too small and convergence is slow. A common starting point is lr=0.01.",
             "key_terms": ["MSE", "cost function", "gradient descent", "learning rate"],
             "difficulty": "intermediate"},
            {"heading": "Multiple Linear Regression",
             "content": "When we have multiple features (x1, x2, ..., xn), the model becomes y = w1*x1 + w2*x2 + ... + wn*xn + b. This is expressed in matrix form as y = Xw + b. The closed-form solution (Normal Equation) is w = (X^T X)^(-1) X^T y. This gives the exact optimal weights without iteration but is expensive for large datasets (O(n^3) for matrix inversion). In practice, gradient descent is preferred for large n. Multicollinearity — when features are highly correlated — inflates weight estimates and makes interpretation unreliable.",
             "key_terms": ["multiple regression", "normal equation", "multicollinearity"],
             "difficulty": "intermediate"},
            {"heading": "Regularization: Ridge and Lasso",
             "content": "Regularization prevents overfitting by penalizing large weights. Ridge regression (L2) adds a penalty term: Loss = MSE + lambda * sum(w_i^2). This shrinks weights toward zero but rarely sets them exactly to zero. Lasso regression (L1) uses: Loss = MSE + lambda * sum(|w_i|). Lasso can drive some weights to exactly zero, performing feature selection. Elastic Net combines both: Loss = MSE + lambda1 * sum(|w_i|) + lambda2 * sum(w_i^2). The regularization strength lambda is a hyperparameter tuned via cross-validation.",
             "key_terms": ["regularization", "ridge", "lasso", "elastic net", "L1", "L2"],
             "difficulty": "intermediate"},
        ],
    },
    {
        "lecture_num": 3,
        "title": "Classification",
        "topic": "classification",
        "sections": [
            {"heading": "Logistic Regression",
             "content": "Despite the name, logistic regression is a classification algorithm. It models the probability that an input belongs to a class using the sigmoid function: P(y=1|x) = 1 / (1 + exp(-(wx + b))). The sigmoid squashes any real number into [0, 1]. We classify as positive if P >= 0.5. Training uses binary cross-entropy loss: L = -[y*log(p) + (1-y)*log(1-p)], minimized via gradient descent. The decision boundary is linear in feature space. Logistic regression is interpretable — each weight indicates how much that feature contributes to the log-odds.",
             "key_terms": ["logistic regression", "sigmoid", "cross-entropy", "decision boundary"],
             "difficulty": "intermediate"},
            {"heading": "Decision Trees",
             "content": "A decision tree splits data into subsets based on feature values, creating a tree of if-then rules. At each node, the algorithm selects the feature and threshold that best separates the classes. Splitting criteria include Gini impurity: Gini = 1 - sum(p_i^2), where p_i is the proportion of class i, and Information Gain based on entropy: Entropy = -sum(p_i * log2(p_i)). Trees are prone to overfitting — a deep tree memorizes the training data. Pruning (limiting tree depth, minimum samples per leaf) is essential. Decision trees are highly interpretable: you can trace the path from root to leaf to explain any prediction.",
             "key_terms": ["decision tree", "Gini impurity", "entropy", "information gain", "pruning"],
             "difficulty": "intermediate"},
            {"heading": "Evaluation Metrics for Classification",
             "content": "Accuracy alone is misleading for imbalanced datasets. Key metrics: Precision = TP/(TP+FP) — of predicted positives, how many are correct? Recall = TP/(TP+FN) — of actual positives, how many did we find? F1 Score = 2 * (Precision * Recall) / (Precision + Recall) — harmonic mean balancing both. The confusion matrix shows TP, FP, TN, FN counts. ROC curve plots True Positive Rate vs False Positive Rate at different thresholds. AUC (Area Under Curve) summarizes overall discriminative ability — 0.5 is random, 1.0 is perfect. For multi-class, use macro/micro/weighted averaging.",
             "key_terms": ["precision", "recall", "F1 score", "confusion matrix", "ROC", "AUC"],
             "difficulty": "intermediate"},
            {"heading": "K-Nearest Neighbors",
             "content": "KNN is a non-parametric algorithm that classifies by majority vote of the k nearest training examples. Distance is typically Euclidean: d(a,b) = sqrt(sum((a_i - b_i)^2)). Key considerations: (1) k selection — small k is noisy and overfits, large k is too smooth and underfits. Use cross-validation to find the optimal k. (2) Feature scaling — KNN is distance-based, so features must be normalized (min-max or z-score). (3) Computational cost — prediction requires computing distance to all training points (O(nd) per query). KD-trees or ball trees speed this up for low-dimensional data.",
             "key_terms": ["KNN", "euclidean distance", "feature scaling", "k selection"],
             "difficulty": "beginner"},
        ],
    },
    {
        "lecture_num": 4,
        "title": "Model Evaluation and Selection",
        "topic": "evaluation",
        "sections": [
            {"heading": "Bias-Variance Tradeoff",
             "content": "Model error decomposes into three parts: Error = Bias^2 + Variance + Irreducible Noise. High bias means the model is too simple and underfits (e.g., linear model for non-linear data). High variance means the model is too complex and overfits (e.g., very deep decision tree). The goal is to find the sweet spot. As model complexity increases, bias decreases but variance increases. Training error always decreases with complexity, but test error follows a U-shape — it decreases then increases. This U-shape is the bias-variance tradeoff in action.",
             "key_terms": ["bias", "variance", "underfitting", "overfitting", "tradeoff"],
             "difficulty": "intermediate"},
            {"heading": "Cross-Validation",
             "content": "Cross-validation estimates how well a model generalizes. K-fold CV: split data into k folds, train on k-1 folds, test on the held-out fold, rotate k times, average the scores. Common choices: k=5 or k=10. Stratified k-fold preserves class proportions in each fold — important for imbalanced data. Leave-one-out CV (LOOCV) sets k=n, giving the least biased estimate but is computationally expensive. Never tune hyperparameters on the test set — use a separate validation set or nested cross-validation: outer loop for evaluation, inner loop for hyperparameter tuning.",
             "key_terms": ["cross-validation", "k-fold", "stratified", "LOOCV", "nested CV"],
             "difficulty": "intermediate"},
            {"heading": "Hyperparameter Tuning",
             "content": "Hyperparameters are settings not learned from data (e.g., learning rate, tree depth, regularization strength). Tuning approaches: (1) Grid search — try all combinations from a predefined grid. Exhaustive but slow. (2) Random search — sample random combinations. Often finds good solutions faster than grid search because not all hyperparameters are equally important. (3) Bayesian optimization — builds a probabilistic model of the objective function and intelligently selects the next combination to try. More efficient for expensive-to-evaluate models. Always report final performance on a held-out test set that was never used during tuning.",
             "key_terms": ["hyperparameter", "grid search", "random search", "Bayesian optimization"],
             "difficulty": "intermediate"},
        ],
    },
    {
        "lecture_num": 5,
        "title": "Ensemble Methods",
        "topic": "ensembles",
        "sections": [
            {"heading": "Bagging and Random Forests",
             "content": "Bagging (Bootstrap Aggregating) trains multiple models on random subsets of the training data (with replacement) and averages their predictions. This reduces variance without increasing bias. Random Forest extends bagging for decision trees: at each split, only a random subset of features (typically sqrt(n_features)) is considered. This decorrelates the individual trees, making the ensemble more robust. Random Forests are hard to overfit by adding more trees — performance plateaus but rarely degrades. Key hyperparameters: n_estimators (number of trees), max_depth, max_features. Feature importance can be derived from how often each feature is used for splitting.",
             "key_terms": ["bagging", "random forest", "bootstrap", "feature importance"],
             "difficulty": "intermediate"},
            {"heading": "Boosting: AdaBoost and Gradient Boosting",
             "content": "Boosting trains models sequentially, each one focusing on the mistakes of the previous. AdaBoost: assign equal weights to all training examples. Train a weak learner (e.g., decision stump). Increase weights on misclassified examples. Repeat. Final prediction is a weighted vote. Gradient Boosting: instead of reweighting examples, each new model fits the residual errors (negative gradient of the loss). XGBoost and LightGBM are optimized implementations with regularization, parallel processing, and handling of missing values. Boosting can overfit if the learning rate is too high or there are too many rounds — use early stopping on a validation set.",
             "key_terms": ["boosting", "AdaBoost", "gradient boosting", "XGBoost", "LightGBM"],
             "difficulty": "advanced"},
            {"heading": "Stacking",
             "content": "Stacking (stacked generalization) uses predictions from multiple base models as input features for a meta-model. Level-0: train diverse models (e.g., logistic regression, random forest, SVM). Level-1: a meta-learner (often logistic regression or linear model) learns the optimal combination of base model predictions. To avoid data leakage, generate level-0 predictions using cross-validation — each training example gets an out-of-fold prediction. Stacking typically provides a small but consistent improvement over the best individual model. It is common in ML competitions but adds complexity in production.",
             "key_terms": ["stacking", "meta-learner", "out-of-fold", "model diversity"],
             "difficulty": "advanced"},
        ],
    },
    {
        "lecture_num": 6,
        "title": "Unsupervised Learning",
        "topic": "unsupervised",
        "sections": [
            {"heading": "K-Means Clustering",
             "content": "K-Means partitions n data points into k clusters by minimizing within-cluster sum of squares. Algorithm: (1) Initialize k centroids randomly. (2) Assign each point to the nearest centroid. (3) Recompute centroids as the mean of assigned points. (4) Repeat steps 2-3 until convergence. K-Means is sensitive to initialization — K-Means++ uses a smarter initialization that spreads centroids apart. Choosing k: use the Elbow method (plot inertia vs k, look for the bend) or the Silhouette score (measures how similar points are to their own cluster vs neighboring clusters). Limitations: assumes spherical clusters, struggles with clusters of different sizes or densities.",
             "key_terms": ["K-Means", "centroids", "elbow method", "silhouette score"],
             "difficulty": "intermediate"},
            {"heading": "Principal Component Analysis",
             "content": "PCA is a dimensionality reduction technique that finds the directions (principal components) of maximum variance in the data. Steps: (1) Standardize features (zero mean, unit variance). (2) Compute the covariance matrix. (3) Find eigenvalues and eigenvectors. (4) Sort by eigenvalue (largest first). (5) Project data onto the top k eigenvectors. The explained variance ratio shows how much information each component captures. A common rule: keep enough components to explain 95% of variance. PCA is useful for visualization (reduce to 2-3 dimensions), noise reduction, and speeding up algorithms on high-dimensional data. PCA assumes linear relationships — for non-linear data, consider t-SNE or UMAP.",
             "key_terms": ["PCA", "dimensionality reduction", "eigenvalues", "explained variance"],
             "difficulty": "advanced"},
            {"heading": "Hierarchical Clustering",
             "content": "Hierarchical clustering builds a tree (dendrogram) of clusters. Agglomerative (bottom-up): start with each point as its own cluster, repeatedly merge the two closest clusters. Divisive (top-down): start with all points in one cluster, recursively split. Linkage criteria determine how 'distance between clusters' is computed: single linkage (minimum distance between any pair), complete linkage (maximum distance), average linkage (mean distance), Ward's method (minimizes total within-cluster variance). Cut the dendrogram at a desired height to get k clusters. Advantage over K-Means: no need to pre-specify k, and the dendrogram reveals cluster structure at multiple granularities.",
             "key_terms": ["hierarchical clustering", "dendrogram", "linkage", "agglomerative"],
             "difficulty": "intermediate"},
        ],
    },
]

print(f"Lectures: {len(LECTURES)}")
total_sections = sum(len(l["sections"]) for l in LECTURES)
print(f"Total sections: {total_sections}")
for lec in LECTURES:
    words = sum(len(s["content"].split()) for s in lec["sections"])
    print(f"  Lecture {lec['lecture_num']}: {lec['title']:40s} | {len(lec['sections'])} sections | {words:>4} words")

## 11. Inspect the Corpus

In [ ]:
print("LECTURE SECTIONS")
print("=" * 110)
for lec in LECTURES:
    for s in lec["sections"]:
        terms = ", ".join(s["key_terms"][:3])
        print(f"  L{lec['lecture_num']} [{s['difficulty']:12s}] {s['heading']:45s} | {terms}")

---

# Part B — Embedding & Indexing

## 12. Build the Vector Index

In [ ]:
# Flatten sections with full metadata
all_sections = []
for lec in LECTURES:
    for s in lec["sections"]:
        all_sections.append({
            "lecture_num": lec["lecture_num"],
            "lecture_title": lec["title"],
            "topic": lec["topic"],
            "heading": s["heading"],
            "content": s["content"],
            "key_terms": s["key_terms"],
            "difficulty": s["difficulty"],
            "section_ref": f"Lecture {lec['lecture_num']}: {lec['title']} > {s['heading']}",
        })

# Embed: section reference + content for better semantic matching
section_texts = [f"{s['section_ref']}\n{s['content']}" for s in all_sections]
section_metadatas = [
    {
        "lecture_num": s["lecture_num"],
        "lecture_title": s["lecture_title"],
        "topic": s["topic"],
        "heading": s["heading"],
        "difficulty": s["difficulty"],
        "section_ref": s["section_ref"],
        "key_terms": ", ".join(s["key_terms"]),
    }
    for s in all_sections
]
section_ids = [f"sec-{i:03d}" for i in range(len(all_sections))]

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("lecture_notes")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="lecture_notes",
    metadata={"hnsw:space": "cosine"},
)

vectors = embeddings.embed_documents(section_texts)
collection.add(
    documents=section_texts,
    embeddings=vectors,
    metadatas=section_metadatas,
    ids=section_ids,
)

print(f"Indexed {collection.count()} sections")
print(f"Embedding dim: {len(vectors[0])}")
print(f"Avg text length: {sum(len(t) for t in section_texts) // len(section_texts)} chars")

---

# Part C — Retrieval Engine

## 13. Core Retrieval Function

In [ ]:
def retrieve(query: str, top_k: int = TOP_K,
             lecture_num: Optional[int] = None,
             topic: Optional[str] = None,
             difficulty: Optional[str] = None) -> list[dict]:
    """Retrieve lecture sections with optional metadata filters."""
    query_vector = embeddings.embed_query(query)
    kwargs = {"query_embeddings": [query_vector], "n_results": top_k}

    conditions = []
    if lecture_num:
        conditions.append({"lecture_num": lecture_num})
    if topic:
        conditions.append({"topic": topic})
    if difficulty:
        conditions.append({"difficulty": difficulty})

    if len(conditions) == 1:
        kwargs["where"] = conditions[0]
    elif len(conditions) > 1:
        kwargs["where"] = {"$and": conditions}

    results = collection.query(**kwargs)
    hits = []
    for i in range(len(results["documents"][0])):
        hits.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "similarity": 1 - results["distances"][0][i],
        })
    return hits


def display_hits(hits: list[dict], label: str = ""):
    if label:
        print(f"\n  [{label}]")
    for i, h in enumerate(hits, 1):
        m = h["metadata"]
        print(f"    {i}. sim={h['similarity']:.3f} | L{m['lecture_num']} [{m['difficulty']:12s}] "
              f"| {m['heading'][:45]}")


def get_section_content(heading: str) -> str:
    """Look up the raw content for a section by heading name."""
    for s in all_sections:
        if s["heading"] == heading:
            return s["content"]
    return ""


# Test
q = "how does gradient descent minimize the cost function"
print(f"Q: {q}")
display_hits(retrieve(q, top_k=3), "Top 3")

## 14. Demonstrate Metadata-Filtered Retrieval

In [ ]:
q = "how to evaluate a model"
print(f"Q: {q}")

print("\n--- No filters ---")
display_hits(retrieve(q, top_k=3), "All")

print("\n--- Filter: Lecture 3 only ---")
display_hits(retrieve(q, top_k=3, lecture_num=3), "Lecture 3")

print("\n--- Filter: beginner difficulty only ---")
display_hits(retrieve(q, top_k=3, difficulty="beginner"), "Beginner")

print("\n--- Filter: ensembles topic ---")
display_hits(retrieve(q, top_k=3, topic="ensembles"), "Ensembles")

---

# Part D — Study Mode 1: Q&A

## 15. Q&A System

In [ ]:
TUTOR_SYSTEM = """You are a patient and knowledgeable study tutor for an ML course.

Rules:
1. Answer ONLY from the provided lecture notes. Do not add external knowledge.
2. Cite every fact as [Ref: Lecture N: Title > Section].
3. Explain clearly — the student is learning, not an expert.
4. Use examples and analogies when they help.
5. If the notes don't cover the topic, say so and suggest which lecture might be relevant."""


def build_context(hits: list[dict]) -> str:
    parts = []
    for h in hits:
        m = h["metadata"]
        content = get_section_content(m["heading"]) or h["text"]
        parts.append(f"[Ref: {m['section_ref']}]\n{content}")
    return "\n\n---\n\n".join(parts)


def study_qa(question: str, **filter_kwargs) -> dict:
    hits = retrieve(question, **filter_kwargs)
    context = build_context(hits)

    prompt = (
        f"A student asks:\n\nQUESTION: {question}\n\n"
        f"LECTURE NOTES:\n{context}\n\n"
        "Return ONLY JSON:\n"
        '{"answer": "your pedagogical answer with [Ref: ...] citations",'
        ' "sources": ["Lecture N: Title > Section"],'
        ' "related_topics": ["topic the student should also review"],'
        ' "confidence": "high|medium|low"}'
    )

    resp = ask(prompt, system=TUTOR_SYSTEM, temperature=TEMP_ANSWER)
    result = parse_json(resp)
    if not result:
        result = {
            "answer": resp,
            "sources": [h["metadata"]["section_ref"] for h in hits[:3]],
            "related_topics": [],
            "confidence": "unknown",
        }
    result["hits"] = hits
    return result


def display_qa(question: str, result: dict):
    print(f"\n{'='*90}")
    print(f"Q: {question}")
    print(f"Confidence: {result.get('confidence', '?')}")
    print(f"{'='*90}")
    print("\nA:")
    wrap_print(str(result.get("answer", "")))
    if result.get("sources"):
        print("\nSources:")
        for src in result["sources"][:4]:
            print(f"  - {src}")
    if result.get("related_topics"):
        print("\nAlso review:")
        for t in result["related_topics"][:3]:
            print(f"  > {t}")


print("Q&A mode ready")

## 16. Q&A Demonstrations

In [ ]:
q = "What is the difference between Ridge and Lasso regularization?"
display_qa(q, study_qa(q))

In [ ]:
q = "When should I use Random Forest vs Gradient Boosting?"
display_qa(q, study_qa(q, topic="ensembles"))

In [ ]:
q = "Why is accuracy a bad metric for imbalanced datasets? What should I use instead?"
display_qa(q, study_qa(q))

In [ ]:
q = "How do I know if my model is overfitting or underfitting?"
display_qa(q, study_qa(q))

---

# Part E — Study Mode 2: Summaries

## 17. Summary Generator

In [ ]:
SUMMARY_SYSTEM = """You are a study notes summarizer. Create concise, study-ready summaries.

Rules:
1. Summarize ONLY content from the provided lecture sections.
2. Use bullet points for key concepts.
3. Include important formulas and definitions.
4. Highlight the most exam-relevant points.
5. Keep it concise — this is a quick review, not re-reading the lecture."""


def generate_summary(topic_query: str, **filter_kwargs) -> dict:
    hits = retrieve(topic_query, top_k=5, **filter_kwargs)
    context = build_context(hits)

    prompt = (
        f"Create a study summary for this topic area:\n\n"
        f"TOPIC: {topic_query}\n\n"
        f"LECTURE MATERIAL:\n{context}\n\n"
        "Return ONLY JSON:\n"
        '{"title": "summary title",'
        ' "key_concepts": ["concept 1", "concept 2"],'
        ' "summary": "concise bullet-point summary for studying",'
        ' "formulas": ["formula 1"],'
        ' "exam_tips": ["what to watch out for"]}'
    )

    resp = ask(prompt, system=SUMMARY_SYSTEM, temperature=TEMP_ANSWER)
    result = parse_json(resp)
    if not result:
        result = {"title": topic_query, "summary": resp,
                  "key_concepts": [], "formulas": [], "exam_tips": []}
    result["hits"] = hits
    return result


def display_summary(result: dict):
    print(f"\n{'='*80}")
    print(f"STUDY SUMMARY: {result.get('title', '?')}")
    print(f"{'='*80}")

    if result.get("key_concepts"):
        print("\nKey Concepts:")
        for c in result["key_concepts"][:8]:
            print(f"  * {c}")

    if result.get("summary"):
        print("\nSummary:")
        wrap_print(str(result["summary"]))

    if result.get("formulas"):
        print("\nImportant Formulas:")
        for f in result["formulas"][:5]:
            print(f"  {f}")

    if result.get("exam_tips"):
        print("\nExam Tips:")
        for t in result["exam_tips"][:4]:
            print(f"  ! {t}")


print("Summary mode ready")

## 18. Generate Summaries

In [ ]:
r = generate_summary("Linear Regression and Regularization", lecture_num=2)
display_summary(r)

In [ ]:
r = generate_summary("Classification algorithms and evaluation", topic="classification")
display_summary(r)

In [ ]:
r = generate_summary("Ensemble methods: bagging, boosting, stacking", topic="ensembles")
display_summary(r)

---

# Part F — Study Mode 3: Quiz

## 19. Quiz Generator

The quiz mode generates questions from the lecture content, then grades student answers — forcing active recall, which is far more effective than passive re-reading.

In [ ]:
QUIZ_SYSTEM = """You are an ML course quiz generator. Create questions to test student understanding.

Rules:
1. Questions must be answerable ONLY from the provided material.
2. Mix difficulty: some factual recall, some conceptual understanding, some application.
3. For multiple-choice, exactly one option must be correct.
4. Include the correct answer and a brief explanation citing the source."""


def generate_quiz(topic_query: str, n_questions: int = 4, **filter_kwargs) -> list[dict]:
    hits = retrieve(topic_query, top_k=5, **filter_kwargs)
    context = build_context(hits)

    prompt = (
        f"Generate {n_questions} quiz questions from these lecture notes.\n\n"
        f"TOPIC: {topic_query}\n\n"
        f"LECTURE MATERIAL:\n{context}\n\n"
        f"Return ONLY a JSON array of {n_questions} questions. Mix types:\n"
        '- MCQ: {"type": "mcq", "question": "...", "options": ["A) ...", "B) ...", "C) ...", "D) ..."], '
        '"correct": "A", "explanation": "...", "difficulty": "easy|medium|hard", '
        '"source": "Lecture N > Section"}\n'
        '- Short answer: {"type": "short_answer", "question": "...", '
        '"correct_answer": "...", "explanation": "...", "difficulty": "easy|medium|hard", '
        '"source": "Lecture N > Section"}'
    )

    resp = ask(prompt, system=QUIZ_SYSTEM, temperature=TEMP_QUIZ)
    questions = parse_json(resp)
    if not questions or not isinstance(questions, list):
        questions = []
    return questions


def display_quiz(questions: list[dict], show_answers: bool = False):
    print(f"\n{'='*80}")
    print(f"QUIZ — {len(questions)} questions")
    print(f"{'='*80}")
    for i, q in enumerate(questions, 1):
        diff = q.get("difficulty", "?")
        print(f"\n  Q{i} [{diff}] ({q.get('type', '?').upper()})")
        wrap_print(f"  {q.get('question', '')}")

        if q.get("type") == "mcq" and q.get("options"):
            for opt in q["options"]:
                print(f"    {opt}")

        if show_answers:
            if q.get("type") == "mcq":
                print(f"  -> Correct: {q.get('correct', '?')}")
            else:
                print(f"  -> Answer: {q.get('correct_answer', '?')}")
            if q.get("explanation"):
                print(f"  -> {q['explanation'][:100]}")
            if q.get("source"):
                print(f"  -> Source: {q['source']}")


print("Quiz mode ready")

## 20. Generate a Quiz — Regression

In [ ]:
quiz_reg = generate_quiz("linear regression, cost function, and regularization", lecture_num=2)
display_quiz(quiz_reg, show_answers=False)

In [ ]:
# Reveal answers
display_quiz(quiz_reg, show_answers=True)

## 21. Generate a Quiz — Classification

In [ ]:
quiz_clf = generate_quiz("classification algorithms and evaluation metrics", topic="classification")
display_quiz(quiz_clf, show_answers=False)

In [ ]:
display_quiz(quiz_clf, show_answers=True)

## 22. Generate a Quiz — Ensembles & Unsupervised

In [ ]:
quiz_ens = generate_quiz("ensemble methods and unsupervised learning", n_questions=4)
display_quiz(quiz_ens, show_answers=False)

In [ ]:
display_quiz(quiz_ens, show_answers=True)

## 23. Answer Grading

Simulate a student providing answers and have the tutor grade them with explanations.

In [ ]:
GRADER_SYSTEM = """You are an ML course grader. Grade student answers against the correct answer and source material.

Be encouraging but accurate. If the student is partially right, give partial credit and explain what's missing."""


def grade_answer(question: str, student_answer: str, correct_answer: str, source_content: str) -> dict:
    prompt = (
        f"Grade this student answer.\n\n"
        f"QUESTION: {question}\n"
        f"CORRECT ANSWER: {correct_answer}\n"
        f"SOURCE MATERIAL: {source_content[:400]}\n\n"
        f"STUDENT ANSWER: {student_answer}\n\n"
        "Return ONLY JSON:\n"
        '{"score": "correct|partial|incorrect",'
        ' "points": N,  // out of 10'
        ' "feedback": "what the student got right/wrong",'
        ' "key_missed": ["concept they should review"]}'
    )
    resp = ask(prompt, system=GRADER_SYSTEM, temperature=TEMP_JUDGE)
    result = parse_json(resp)
    if not result:
        result = {"score": "unknown", "points": 0, "feedback": resp, "key_missed": []}
    return result


# Simulate grading three student answers
test_answers = [
    {
        "question": "What is the difference between Ridge and Lasso regularization?",
        "student": "Ridge uses L2 penalty which shrinks weights, and Lasso uses L1 penalty which can make weights exactly zero so it does feature selection.",
        "correct": "Ridge (L2) adds sum of squared weights as penalty — shrinks toward zero but doesn't eliminate. Lasso (L1) adds sum of absolute weights — can set weights to exactly zero, performing feature selection.",
        "source_heading": "Regularization: Ridge and Lasso",
    },
    {
        "question": "Why is accuracy misleading for imbalanced datasets?",
        "student": "Because if 99% of data is one class, predicting that class always gives 99% accuracy.",
        "correct": "Accuracy can be high by always predicting the majority class. Use precision, recall, F1 score instead.",
        "source_heading": "Evaluation Metrics for Classification",
    },
    {
        "question": "How does K-Means clustering work?",
        "student": "It groups data into clusters.",
        "correct": "Initialize k centroids, assign each point to nearest centroid, recompute centroids as mean, repeat until convergence.",
        "source_heading": "K-Means Clustering",
    },
]

print("ANSWER GRADING")
print("=" * 80)
for ta in test_answers:
    source_content = get_section_content(ta["source_heading"])
    grade = grade_answer(ta["question"], ta["student"], ta["correct"], source_content)
    print(f"\n  Q: {ta['question'][:65]}")
    print(f"  Student: {ta['student'][:80]}")
    score_str = grade.get('score', '?')
    points = grade.get('points', '?')
    print(f"  Grade: {score_str} ({points}/10)")
    if grade.get("feedback"):
        print(f"  Feedback: {str(grade['feedback'])[:100]}")
    if grade.get("key_missed"):
        missed = grade["key_missed"]
        if isinstance(missed, list) and missed:
            print(f"  Review: {', '.join(str(m) for m in missed[:3])}")

---

# Part G — Retrieval Evaluation

## 24. Ground-Truth Evaluation Set

In [ ]:
EVAL_SET = [
    {"query": "what are the three types of machine learning",
     "expected_heading": "Types of Machine Learning",
     "filters": {"lecture_num": 1}},
    {"query": "steps in a machine learning project",
     "expected_heading": "The ML Pipeline",
     "filters": {}},
    {"query": "how does gradient descent minimize error",
     "expected_heading": "Cost Function and Gradient Descent",
     "filters": {"lecture_num": 2}},
    {"query": "ridge vs lasso penalty term",
     "expected_heading": "Regularization: Ridge and Lasso",
     "filters": {"topic": "regression"}},
    {"query": "sigmoid function and log loss for classification",
     "expected_heading": "Logistic Regression",
     "filters": {"topic": "classification"}},
    {"query": "gini impurity and information gain for splitting",
     "expected_heading": "Decision Trees",
     "filters": {}},
    {"query": "precision recall F1 confusion matrix",
     "expected_heading": "Evaluation Metrics for Classification",
     "filters": {}},
    {"query": "how to choose the number of neighbors in KNN",
     "expected_heading": "K-Nearest Neighbors",
     "filters": {"topic": "classification"}},
    {"query": "underfitting vs overfitting model complexity",
     "expected_heading": "Bias-Variance Tradeoff",
     "filters": {"topic": "evaluation"}},
    {"query": "k-fold stratified cross validation",
     "expected_heading": "Cross-Validation",
     "filters": {}},
    {"query": "grid search vs random search vs bayesian",
     "expected_heading": "Hyperparameter Tuning",
     "filters": {}},
    {"query": "random forest feature importance bagging",
     "expected_heading": "Bagging and Random Forests",
     "filters": {"topic": "ensembles"}},
    {"query": "XGBoost boosting residuals",
     "expected_heading": "Boosting: AdaBoost and Gradient Boosting",
     "filters": {}},
    {"query": "stacking meta-learner out-of-fold",
     "expected_heading": "Stacking",
     "filters": {}},
    {"query": "K-Means elbow method silhouette score",
     "expected_heading": "K-Means Clustering",
     "filters": {}},
    {"query": "PCA eigenvalues explained variance ratio",
     "expected_heading": "Principal Component Analysis",
     "filters": {}},
]

print(f"Evaluation set: {len(EVAL_SET)} query-section pairs")

## 25. Run Evaluation

In [ ]:
def run_eval(eval_set: list[dict], use_filters: bool, top_k: int = 5) -> dict:
    hits_at_k = 0
    recip_ranks = []
    details = []

    for item in eval_set:
        fkw = item["filters"] if use_filters else {}
        hits = retrieve(item["query"], top_k=top_k, **fkw)
        headings = [h["metadata"]["heading"] for h in hits]

        found_rank = None
        for rank, h in enumerate(headings, 1):
            if h == item["expected_heading"]:
                found_rank = rank
                break

        if found_rank is not None:
            hits_at_k += 1
            recip_ranks.append(1.0 / found_rank)
        else:
            recip_ranks.append(0.0)

        details.append({
            "query": item["query"],
            "expected": item["expected_heading"],
            "top_3": headings[:3],
            "hit": found_rank is not None,
            "rank": found_rank,
        })

    n = len(eval_set)
    return {
        "recall_at_k": hits_at_k / n,
        "mrr": sum(recip_ranks) / n,
        "n": n,
        "k": top_k,
        "details": details,
    }


eval_unf = run_eval(EVAL_SET, use_filters=False, top_k=5)
eval_flt = run_eval(EVAL_SET, use_filters=True, top_k=5)

print("RETRIEVAL EVALUATION")
print("=" * 55)
print(f"{'Metric':<20} {'Unfiltered':>12} {'Filtered':>12}")
print("-" * 55)
print(f"{'Recall@5':<20} {eval_unf['recall_at_k']:>11.1%} {eval_flt['recall_at_k']:>11.1%}")
print(f"{'MRR':<20} {eval_unf['mrr']:>11.3f} {eval_flt['mrr']:>11.3f}")
print(f"{'Questions':<20} {eval_unf['n']:>12} {eval_flt['n']:>12}")

In [ ]:
print("\nPER-QUERY RESULTS (filtered)")
print("=" * 90)
for d in eval_flt["details"]:
    mark = "HIT" if d["hit"] else "MISS"
    rank_s = f"@{d['rank']}" if d["hit"] else "---"
    print(f"  [{mark}] {rank_s:>4} | {d['query'][:45]:45s} -> {d['expected'][:30]}")

In [ ]:
print("RECALL AT DIFFERENT K")
print("=" * 55)
print(f"{'k':>3} | {'Unfilt R@k':>12} {'Unfilt MRR':>12} | {'Filt R@k':>12} {'Filt MRR':>12}")
print("-" * 55)
for k in [1, 3, 5]:
    u = run_eval(EVAL_SET, use_filters=False, top_k=k)
    f = run_eval(EVAL_SET, use_filters=True, top_k=k)
    print(f"  {k:>1} | {u['recall_at_k']:>11.1%} {u['mrr']:>11.3f} | {f['recall_at_k']:>11.1%} {f['mrr']:>11.3f}")

## 26. Q&A Quality — LLM-as-Judge

In [ ]:
JUDGE_SYSTEM = "You evaluate study tutor answers for educational quality."

JUDGE_PROMPT = """Evaluate this study tutor answer.

QUESTION: {question}
ANSWER: {answer}

Score each dimension 1-5:
- correctness: factually accurate per the source material?
- pedagogical_quality: explains concepts clearly for a student?
- citations: references specific lecture sections?
- completeness: covers the key points needed to answer?

Return ONLY JSON:
{{"correctness": N, "pedagogical_quality": N, "citations": N,
  "completeness": N, "overall": N, "notes": "explanation"}}"""

judge_questions = [
    "What is the difference between Ridge and Lasso regularization?",
    "How do Random Forests prevent overfitting?",
    "Why is accuracy misleading for imbalanced data?",
]

print("Q&A QUALITY (LLM-as-Judge)")
print("=" * 80)
for q in judge_questions:
    result = study_qa(q)
    resp = ask(
        JUDGE_PROMPT.format(
            question=q,
            answer=str(result.get("answer", ""))[:400],
        ),
        system=JUDGE_SYSTEM,
        temperature=TEMP_JUDGE,
    )
    scores = parse_json(resp)
    if scores:
        print(f"\n  Q: {q[:60]}")
        for dim in ["correctness", "pedagogical_quality", "citations", "completeness"]:
            val = scores.get(dim, "?")
            bar = "*" * int(val) if isinstance(val, (int, float)) else "?"
            print(f"    {dim:22s}: {val}/5 {bar}")
        print(f"    {'overall':22s}: {scores.get('overall', '?')}/5")
        if scores.get("notes"):
            print(f"    Notes: {scores['notes'][:80]}")

---

## 27. Edge Cases

In [ ]:
print("EDGE CASE 1: Question about a topic not covered")
q = "How does a transformer model work with self-attention?"
display_qa(q, study_qa(q))

print("\n" + "="*90)
print("EDGE CASE 2: Very vague question")
q = "What should I study?"
display_qa(q, study_qa(q))

print("\n" + "="*90)
print("EDGE CASE 3: Cross-lecture concept question")
q = "How does regularization relate to the bias-variance tradeoff?"
display_qa(q, study_qa(q))

## 28. Common Mistakes

| Mistake | Why It's Wrong | Better Approach |
|---------|---------------|----------------|
| Chunking lectures by character count | Splits mid-concept; chunks lose context | Split by section headings |
| No metadata beyond source doc | Can't filter by lecture, topic, or difficulty | Store lecture_num, topic, difficulty, key_terms |
| Same prompt for Q&A, summary, and quiz | Each task needs different instructions and output format | Separate system prompts per mode |
| Quiz questions not grounded in material | LLM generates questions from general knowledge | Force retrieval first, generate only from context |
| No answer grading pipeline | Students get questions but no feedback | Build a grader that compares against source material |
| Single difficulty level for quizzes | Beginners get overwhelmed, advanced students get bored | Tag difficulty per section; match quiz difficulty |

## 29. Production Improvement Ideas

1. **Spaced repetition** — track which questions the student gets wrong and re-quiz on those topics at increasing intervals
2. **Concept graph** — build a prerequisite graph (e.g., gradient descent requires calculus) and suggest review order
3. **PDF/PPTX ingestion** — parse real lecture slides using pypdf or python-pptx
4. **Adaptive difficulty** — start with easy questions, increase difficulty based on performance
5. **LaTeX rendering** — render formulas from lecture notes properly in the quiz
6. **Progress tracking** — dashboard showing which topics the student has mastered vs needs review
7. **Multiple embedding models** — compare general vs educational domain embeddings

## 30. Exercises

### Exercise 1
Add a 7th lecture on Neural Networks (cover perceptron, backpropagation, activation functions). Write 3 evaluation pairs and verify retrieval still works across all 7 lectures.

### Exercise 2
Implement a `study_plan` function: given a student's quiz results (which questions they got wrong), retrieve the corresponding lecture sections and generate a prioritized study plan.

### Exercise 3
Build a comparison mode: given two concepts (e.g., "Ridge vs Lasso"), retrieve both sections, and generate a structured comparison table highlighting similarities, differences, and when to use each.

### Mini Challenge
Implement **adaptive quizzing**: start with easy questions, track correct/incorrect, and dynamically select the next question's difficulty. After 10 questions, show a report of mastery level per topic.

In [ ]:
print("SESSION SUMMARY")
print("=" * 60)
print(f"Lectures indexed: {len(LECTURES)}")
print(f"Total sections: {len(all_sections)}")
total_words = sum(len(s['content'].split()) for s in all_sections)
print(f"Total words: {total_words:,}")
print(f"Evaluation pairs: {len(EVAL_SET)}")
print(f"\nRetrieval (Recall@5 / MRR):")
print(f"  Unfiltered: {eval_unf['recall_at_k']:.1%} / {eval_unf['mrr']:.3f}")
print(f"  Filtered:   {eval_flt['recall_at_k']:.1%} / {eval_flt['mrr']:.3f}")
print(f"\nStudy modes built:")
print(f"  - Q&A with lecture citations and related topic suggestions")
print(f"  - Summary generation with key concepts and exam tips")
print(f"  - Quiz generation (MCQ + short answer) with difficulty levels")
print(f"  - Answer grading with feedback and review suggestions")
print(f"  - Retrieval evaluation (Recall@k, MRR)")
print(f"  - LLM-as-judge for pedagogical quality")

## 31. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Educational RAG differs from general RAG** — tone, metadata, and generation goals are all different |
| 2 | **Section-based chunking** aligned with lecture structure gives better retrieval than character splitting |
| 3 | **Lecture metadata** (number, topic, difficulty) enables targeted retrieval per study mode |
| 4 | **Three study modes** from one index — Q&A, summary, and quiz are different prompts over the same retrieval |
| 5 | **Quiz generation + grading** implements active recall, the most effective study technique |
| 6 | **Answer grading** needs the source material, not just the correct answer — context makes feedback richer |
| 7 | **Cross-lecture questions** work when top-k retrieves from multiple lectures |
| 8 | **Difficulty metadata** enables adaptive quizzing — beginner students get easier questions first |
| 9 | **Evaluation with ground-truth pairs** is essential to ensure retrieval quality as the corpus grows |
| 10 | The tutor should **cite lecture sources** so students know exactly where to review |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*